# Millimeter Wave Radar Configuration Parameter Validity Check

1. The main purpose is to verify whether the parameters in the mmWave radar cfg file and the DCA data acquisition board cf.json file are correctly configured.
2. The constraint conditions for the parameters are based on the characteristics of the AWR2243 device. For details, please refer to the AWR2243 datasheet, mmWave SDK user manual, and chirp programming manual.
3. If the parameters meet the constraint conditions, debugging information will be output in cyan; if not, it will be output in purple or yellow.
4. Note that the constraint conditions in this program are not completely accurate. Therefore, even if all parameters meet the constraints, there is still a chance that the system may not run properly in special cases.

### Load Related Libraries

In [1]:

from color_log.clog import ColorLog, MODE_CONSOLE_LOG  # pip install py-color-log
import math
import json
from mmwave.dataloader import DCA1000
log = ColorLog(name='radar_param_logger', mode=MODE_CONSOLE_LOG)

### Read Configuration Files
Configuration file information can be read from the cfg file using the DCA1000.read_config() method, or you can manually input the ADC_PARAMS dictionary. Please refer to the "Modify Configuration File" section for its structure.

In [2]:
# Read mmWave radar cfg configuration file
_, _, ADC_PARAMS, CFG_PARAMS = DCA1000.AWR2243_read_config("configFiles/AWR2243_mmwaveconfig.txt")
print(ADC_PARAMS)
print(CFG_PARAMS)

# Read DCA acquisition board cf.json configuration file
with open('configFiles/cf.json') as cf_json:
    cf_json_load = json.loads(cf_json.read())
    print(cf_json_load)

{'chirps': 128, 'rx': 4, 'tx': 1, 'samples': 256, 'IQ': 2, 'bytes': 2, 'startFreq': 77.00025880336761, 'idleTime': 100.0, 'adc_valid_start_time': 6.0, 'rampEndTime': 60.0, 'freq_slope': 29.98173236846924, 'txStartTime': 0.0, 'sample_rate': 10000, 'frame_periodicity': 70.0}
{'txAntMask': 7, 'numTxChan': 3, 'rxAntMask': 15, 'numRxChan': 4, 'lvdsBW': 600, 'numlaneEn': 4}
{'DCA1000Config': {'dataLoggingMode': 'raw', 'dataTransferMode': 'LVDSCapture', 'dataCaptureMode': 'ethernetStream', 'lvdsMode': 1, 'dataFormatMode': 3, 'packetDelay_us': 25, 'ethernetConfig': {'DCA1000IPAddress': '192.168.33.180', 'DCA1000ConfigPort': 4096, 'DCA1000DataPort': 4098}, 'ethernetConfigUpdate': {'systemIPAddress': '192.168.33.30', 'DCA1000IPAddress': '192.168.33.180', 'DCA1000MACAddress': '12.34.56.78.90.12', 'DCA1000ConfigPort': 4096, 'DCA1000DataPort': 4098}, 'captureConfig': {'fileBasePath': 'D:\\git\\ar1xxx_mmwavestudio_bitbucket\\mmWaveStudioPkg\\mmWaveStudio_Internal\\PostProc', 'filePrefix': 'lua_check

### Constraint Calculation
If the parameters meet the constraint conditions, debugging information will be output in cyan; if not, it will be output in purple or yellow. Calculated information such as distance and velocity resolution will be output in black.

In [3]:
# Constraint conditions mainly refer to Programming Chirp Parameters in TI Radar Devices (Rev. A).pdf

# Check if the slope of the linear frequency modulated chirp pulse meets the constraint: freqSlope <= 266 MHz/us
if (ADC_PARAMS['startFreq'] >= 76):
    CLI_FREQ_SCALE_FACTOR = 3.6 # 77GHz
else:
    CLI_FREQ_SCALE_FACTOR = 2.7 # 60GHz
mmwStartFreqConst = math.trunc(ADC_PARAMS['startFreq'] * (1<<26) / CLI_FREQ_SCALE_FACTOR )
startFreqConst_actual = mmwStartFreqConst * (CLI_FREQ_SCALE_FACTOR / (1<<26))
log.debug('mmwStartFreqConst: %d, startFreqConst_actual: %f(GHz)' % (mmwStartFreqConst, startFreqConst_actual))
mmwFreqSlopeConst = math.trunc(ADC_PARAMS['freq_slope'] * (1<<26) / (CLI_FREQ_SCALE_FACTOR*1e3*900) )
freqSlopeConst_actual = mmwFreqSlopeConst * ((CLI_FREQ_SCALE_FACTOR*1e3*900) / (1<<26))
logstr = 'mmwFreqSlopeConst: %d, freqSlopeConst_actual: %f(MHz/us) must <= 266' % (mmwFreqSlopeConst, freqSlopeConst_actual)
if(freqSlopeConst_actual <= 266.0215):
    log.info(logstr)
else:
    log.error(logstr)

# Check if the frame repetition period meets the constraint: 300(us) <= frame_periodicity <= 1.342(s) (refer to DFP manual)
mmwFramePeriodicityConst = math.trunc(ADC_PARAMS['frame_periodicity'] / (5*1e-6) )
framePeriodicityConst_actual = mmwFramePeriodicityConst * 5*1e-6
logstr = ('mmwFramePeriodicityConst: %d, 0.3(ms) <= frame periodicity: %.3f(ms) <= 1342(ms)' % (mmwFramePeriodicityConst, framePeriodicityConst_actual))
if(0.3 <= framePeriodicityConst_actual and framePeriodicityConst_actual <= 1342):
    log.info(logstr)
else:
    log.error(logstr)

# Check if the active frame period (Tf=N*Tc) meets the constraint: active_frame_time < framePeriodicityConst_actual
chirp_repetition_period = ADC_PARAMS['tx'] * (ADC_PARAMS['idleTime'] + ADC_PARAMS['rampEndTime']) # Tc (us), chirp pulse interval period
log.debug('chirp repetition period: %.2f(us) -> max single tx chirp freq: %.2f(kHz)' % (chirp_repetition_period, 1e3/chirp_repetition_period))
active_frame_time = ADC_PARAMS['chirps'] * chirp_repetition_period / 1e3 # Tf=N*Tc (ms)
logstr = 'active frame time: %.3f(ms) must < frame periodicity: %.3f(ms)' % (active_frame_time, framePeriodicityConst_actual)
if(active_frame_time < framePeriodicityConst_actual):
    log.info(logstr)
    all_tx_chirp_freq = ADC_PARAMS['tx'] * ADC_PARAMS['chirps'] / framePeriodicityConst_actual
    log.debug('All tx chirp repetition period: %.2f(us) -> All tx chirp freq: %.2f(kHz)' % (1e3/all_tx_chirp_freq, all_tx_chirp_freq))
else:
    log.error(logstr)

# Check if the sampling bandwidth meets the constraint: sampling bandwidth < effective pulse ramp bandwidth (determined by chirp pulse ramp duration and slope)
adc_sample_time = 1e3 * ADC_PARAMS['samples'] / ADC_PARAMS['sample_rate']
max_adc_sample_time = ADC_PARAMS['rampEndTime'] - ADC_PARAMS['adc_valid_start_time'] - 0.15 # Not subtracting 0.15 causes distortion at the end sampling point
log.debug('adc start time: %.2f(us) adc sample time: %.2f(us) max adc sample time: %.2f(us) ramp end time: %.2f(us)' % (ADC_PARAMS['adc_valid_start_time'], adc_sample_time, max_adc_sample_time, ADC_PARAMS['rampEndTime']))
adc_sample_sweep_bandwidth = freqSlopeConst_actual * 1e6 * adc_sample_time
max_adc_sweep_bandwidth = freqSlopeConst_actual * 1e6 * max_adc_sample_time
logstr = ('adc_sample_sweep_bandwidth: %.2f(MHz) must < max_adc_sweep_bandwidth: %.2f(MHz)' % (adc_sample_sweep_bandwidth/1e6, max_adc_sweep_bandwidth/1e6))
if(adc_sample_sweep_bandwidth < max_adc_sweep_bandwidth):
    log.info(logstr)
else:
    log.error(logstr)

# Check if the total pulse ramp bandwidth meets the constraint: total pulse ramp bandwidth < 4GHz
max_sweep_bandwidth = (81 - startFreqConst_actual) * 1e9
total_ramp_sweep_bandwidth = freqSlopeConst_actual * 1e6 * ADC_PARAMS['rampEndTime']
logstr = ('total_ramp_sweep_bandwidth: %.2f(MHz) must <= %.0f(MHz)' % (total_ramp_sweep_bandwidth/1e6, max_sweep_bandwidth/1e6))
if(total_ramp_sweep_bandwidth/1e6 <= max_sweep_bandwidth/1e6):
    log.info(logstr)
else:
    log.error(logstr)

# Check if the idle time between two pulses meets the constraint: ADC_PARAMS['idleTime'] >= synthesizerRampDownTime
if(total_ramp_sweep_bandwidth/1e6 < 1000):
    synthesizerRampDownTime = 2
elif(total_ramp_sweep_bandwidth/1e6 < 2000):
    synthesizerRampDownTime = 3.5
elif(total_ramp_sweep_bandwidth/1e6 < 3000):
    synthesizerRampDownTime = 5
else:
    synthesizerRampDownTime = 7
logstr = 'idle time: %.3f(us) must >= Synthesizer Ramp Down Time: %.3f(us)' % (ADC_PARAMS['idleTime'], synthesizerRampDownTime)
if(ADC_PARAMS['idleTime'] >= synthesizerRampDownTime):
    log.info(logstr)
else:
    log.error(logstr)

# Calculate and display parameters such as range and velocity resolution
midFreq = startFreqConst_actual * 1e9 + (ADC_PARAMS['adc_valid_start_time'] + adc_sample_time/2) * freqSlopeConst_actual * 1e6
midFreqWaveLenth = 3e8 / midFreq

numDopplerBins = 2 ** math.ceil(math.log2(ADC_PARAMS['chirps']))
numRangeBins = 2 ** math.ceil(math.log2(ADC_PARAMS['samples']))

rangeResolutionMeters = 3e8 / (2 * adc_sample_sweep_bandwidth)
rangeIdxToMeters = (3e8 * ADC_PARAMS['sample_rate'] * 1e3) / (2 * freqSlopeConst_actual * 1e12 * numRangeBins)
dopplerResolutionMps = midFreqWaveLenth / (2 * numDopplerBins * chirp_repetition_period * 1e-6)
maxRange = (300 * 0.8 * ADC_PARAMS['sample_rate']) / (2 * freqSlopeConst_actual * 1e3)
# maxRange = 0.8 * rangeResolutionMeters * ADC_PARAMS['samples']
maxVelocity = midFreqWaveLenth / (4 * chirp_repetition_period * 1e-6)

log.debug('max range: %.2f(m)' % maxRange)
log.debug('range resolution: %.4f(m), range interbin resolution: %.4f(m)' % (rangeResolutionMeters, rangeIdxToMeters))
log.debug('max velocity: %.2f(m/s)' % maxVelocity)
log.debug('velocity resolution: %.2f(m/s)' % (dopplerResolutionMps))

# Check if the maximum beat frequency signal IF bandwidth of the mixer output meets the constraint: maximum_beat_frequency <= 20(MHz)
maximum_beat_frequency = 2e6 * freqSlopeConst_actual * maxRange / 3e8
logstr = ('max beat frequency: %.2f(MHz) must <= max I/F bandwidth 20(MHz)' % maximum_beat_frequency)
if(maximum_beat_frequency <= 20):
    log.info(logstr)
else:
    log.error(logstr)

# Check if the sampling rate meets the constraint: maximum_sampling_frequency <= 22.5(MHz)
logstr = ('sample rate: %.2f(MHz) must <= max sampling frequency 22.5(MHz)' % (ADC_PARAMS['sample_rate']/1e3))
if(ADC_PARAMS['sample_rate']/1e3 <= 22.5):
    log.info(logstr)
else:
    log.error(logstr)

# Check if the radar cube memory size required by the on-chip DSP meets the constraint: radar_cube_size <= 1024(KB)
# radar_cube_size = numRangeBins * ADC_PARAMS['chirps'] * ADC_PARAMS['tx'] * ADC_PARAMS['rx'] * 4 / 1024
# logstr = ('radar cube size: %.2f(KB) must <= 1024(KB)' % radar_cube_size)
# if(radar_cube_size <= 1024):
#     log.info(logstr)
# else:
#     log.error(logstr)

# Check if the amount of raw IQ data meets the constraint: LVDS Data Size Per Chirp <= max Send Bytes Per Chirp
LVDSDataSizePerChirp = ADC_PARAMS['samples'] * ADC_PARAMS['rx'] * ADC_PARAMS['IQ'] * ADC_PARAMS['bytes'] + 52
LVDSDataSizePerChirp = math.ceil(LVDSDataSizePerChirp / 256) * 256
maxSendBytesPerChirp = (ADC_PARAMS['idleTime'] + ADC_PARAMS['rampEndTime']) * CFG_PARAMS['numlaneEn'] * CFG_PARAMS['lvdsBW'] / 8
logstr = ("LVDS Data Size Per Chirp: %d(Bytes) must <= max Send Bytes Per Chirp: %d(Bytes)" % (LVDSDataSizePerChirp, maxSendBytesPerChirp))
if(LVDSDataSizePerChirp <= maxSendBytesPerChirp):
    log.info(logstr)
else:
    log.error(logstr)

# Check if the FPGA throughput (related to delay) meets the constraint: FPGA Packet Delay <= min required Packet Delay
BYTES_IN_PACKET = 1456  # Data in payload per packet from FPGA
BYTES_OF_PACKET = 1466  # payload size per packet from FPGA
UDP_PACKET_OVERHEAD = 8
IP_OVERHEAD = 20
ETH_OVERHEAD = 14
SizeInMBperSec = ADC_PARAMS['samples'] * ADC_PARAMS['rx'] * ADC_PARAMS['IQ'] * ADC_PARAMS['bytes'] * ADC_PARAMS['tx'] * ADC_PARAMS['chirps'] * (1e-3 / framePeriodicityConst_actual)
overheadRatio = (BYTES_OF_PACKET + UDP_PACKET_OVERHEAD + IP_OVERHEAD + ETH_OVERHEAD) / BYTES_IN_PACKET
FPGA_throughput = 8 * SizeInMBperSec * overheadRatio
log.debug('FPGA throughput: %.2f(Mbps)' % (FPGA_throughput))
minPacketDelay = 138.57 * math.exp(-FPGA_throughput / 179.157) + 2.73 # Fitted according to DCA1000EVA manual data
logstr = ("FPGA Packet Delay: %d(us) should <= min required Packet Delay: %.2f(us)" % (cf_json_load['DCA1000Config']['packetDelay_us'], minPacketDelay))
if(cf_json_load['DCA1000Config']['packetDelay_us'] <= minPacketDelay):
    log.info(logstr)
else:
    log.warn(logstr)

print('current params:', ADC_PARAMS)

2025-08-23 19:12:13 [DEBUG] - mmwStartFreqConst: 1435388860, startFreqConst_actual: 77.000259(GHz)
2025-08-23 19:12:13 [INFO] - mmwFreqSlopeConst: 621, freqSlopeConst_actual: 29.981732(MHz/us) must <= 266
2025-08-23 19:12:13 [INFO] - mmwFramePeriodicityConst: 14000000, 0.3(ms) <= frame periodicity: 70.000(ms) <= 1342(ms)
2025-08-23 19:12:13 [DEBUG] - chirp repetition period: 160.00(us) -> max single tx chirp freq: 6.25(kHz)
2025-08-23 19:12:13 [INFO] - active frame time: 20.480(ms) must < frame periodicity: 70.000(ms)
2025-08-23 19:12:13 [DEBUG] - All tx chirp repetition period: 546.88(us) -> All tx chirp freq: 1.83(kHz)
2025-08-23 19:12:13 [DEBUG] - adc start time: 6.00(us) adc sample time: 25.60(us) max adc sample time: 53.85(us) ramp end time: 60.00(us)
2025-08-23 19:12:13 [INFO] - adc_sample_sweep_bandwidth: 767.53(MHz) must < max_adc_sweep_bandwidth: 1614.52(MHz)
2025-08-23 19:12:13 [INFO] - total_ramp_sweep_bandwidth: 1798.90(MHz) must <= 4000(MHz)
2025-08-23 19:12:13 [INFO] - id

current params: {'chirps': 128, 'rx': 4, 'tx': 1, 'samples': 256, 'IQ': 2, 'bytes': 2, 'startFreq': 77.00025880336761, 'idleTime': 100.0, 'adc_valid_start_time': 6.0, 'rampEndTime': 60.0, 'freq_slope': 29.98173236846924, 'txStartTime': 0.0, 'sample_rate': 10000, 'frame_periodicity': 70.0}


### Modify Configuration File Parameters
You can try modifying some parameters here and rerun the "Constraint Calculation" code block to verify the validity of the parameters.

In [13]:
# Modify parameters here and then run the previous calculation cell to check the effect
ADC_PARAMS = {
    'chirps': 128,               # Corresponds to loopCount in txt
    'rx': 4, 
    'tx': 3, 
    'samples': 256,              # Corresponds to numAdcSamples in txt, rlFrameCfg_t is twice rlProfileCfg_t
    'IQ': 2, 
    'bytes': 2, 
    'startFreq': 76,             # Corresponds to startFreqConst in txt, remember to convert to mmwStartFreqConst
    'idleTime': 7.0,             # Corresponds to idleTimeConst in txt, remember to use 10ns units (idleTimeConst=700, equals 7us)
    'adc_valid_start_time': 4.0, # Corresponds to adcStartTimeConst in txt, remember to use 10ns units (adcStartTimeConst=400, equals 4us)
    'rampEndTime': 15.53,        # Corresponds to rampEndTime in txt, remember to use 10ns units (rampEndTime=1553, equals 15.53us)
    'freq_slope': 265.008,       # Corresponds to freqSlopeConst in txt, remember to convert to mmwFreqSlopeConst
    'txStartTime': 1,            # Corresponds to txStartTime in txt, remember to use 10ns units (txStartTime=100, equals 1us)
    'sample_rate': 22500,        # Corresponds to digOutSampleRate in txt
    'frame_periodicity': 8.952   # Corresponds to periodicity in txt, remember to convert to mmwFramePeriodicityConst
}

### Test
Test the functionality of the ColorLog module

In [12]:
from color_log.clog import ColorLog
log = ColorLog(name='radar_param_logger', mode=MODE_CONSOLE_LOG)

log.debug('This is a DEBUG log, white')
log.info('This is an INFO log, cyan-green')
log.warn('This is a WARN log, yellow')
log.error('This is an ERROR log, purple')
try:
    a = 1 / 0
except:
    log.critical('This is a CRITICAL log, red')

2025-08-12 15:19:10 [DEBUG] - This is a DEBUG log, white
2025-08-12 15:19:10 [INFO] - This is an INFO log, cyan-green
2025-08-12 15:19:10 [WARNING] - This is a WARN log, yellow
2025-08-12 15:19:10 [ERROR] - This is an ERROR log, purple
2025-08-12 15:19:10 [CRITICAL] - This is a CRITICAL log, red
2025-08-12 15:19:10 [CRITICAL] - Traceback (most recent call last):
  File "C:\Users\ליאור\AppData\Local\Temp\ipykernel_13788\3270925969.py", line 9, in <module>
    a = 1 / 0
        ~~^~~
ZeroDivisionError: division by zero

